# 🔊 Inferencia — HybridCNN Audio Classifier

Carga el modelo entrenado y predice sobre audios `.wav` / `.mp3` usando **exactamente el mismo pipeline** que `preprocess.py → main()` (sr=44100, n_mels=128, n_mfcc=13, n_fft=2048, hop=512, sin `target_duration`).

## 1 · Importaciones

In [36]:
from __future__ import annotations

import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")

from src.utils.config import *
from src.models.hybrid_cnn_v3 import ImprovedMFCCCNN
from src.data.preprocess import Preprocess, PreprocessConfig  # ← pipeline centralizado

print("✅ Importaciones completadas")


✅ Importaciones completadas


## 2 · Configuración

In [37]:
# ─────────────────────────────────────────────
# 🔧 RUTAS
# ─────────────────────────────────────────────
MODEL_PATH         = FINAL_MODEL_DIR / "best_no_alertable_v4.pt"
# MODEL_PATH       = CHECKPOINT_DIR /"human_label"/ "checkpoint_epoch_22.pt"   # ← descomenta si usas checkpoint
# MODEL_PATH       = CHECKPOINT_DIR /"human_label_V3"/ "checkpoint_epoch_24.pt"   # ← descomenta si usas checkpoint
# MODEL_PATH       = FINAL_MODEL_DIR / "model_total_V4_32.pt"   # ← descomenta si usas checkpoint

LABEL_MAPPING_PATH = LABEL_MAPPING["human_label_no_alertable2"]

AUDIO_FOLDER       = TEST_AUDIO_FOLDER / "no_alertables"   # carpeta con .wav / .mp3

# # ─────────────────────────────────────────────
# # 🎛️ PARÁMETROS DE AUDIO  (igual que preprocess.py → main())
# # ─────────────────────────────────────────────
# SAMPLE_RATE    = 44100
# N_MELS         = 128
# N_MFCC         = 13      # igual que preprocess
# N_FFT          = 2048
# HOP_LENGTH     = 512
# PEAK_TARGET    = 0.99    # normalización de pico
# ─────────────────────────────────────────────
# 🎛️ PARÁMETROS DE AUDIO  (igual que preprocess.py → main())
# ─────────────────────────────────────────────
SAMPLE_RATE    = 16000
N_MELS         = 128
N_MFCC         = 13
N_FFT          = 1024
HOP_LENGTH     = 160
PEAK_TARGET    = 0.99
# ─────────────────────────────────────────────
# 🧠 MODELO
# ─────────────────────────────────────────────
MODE           = "mel_waveform"   # "mel_only" | "mfcc_only" | "mel_mfcc" | "mel_waveform"
DROPOUT        = 0.30
AUGMENT_INFERENCE=True
# ─────────────────────────────────────────────
# 🖥️ INFERENCIA
# ─────────────────────────────────────────────
TOP_K          = 5       # cuántas predicciones mostrar por audio

print(f"📂 Carpeta de audios : {AUDIO_FOLDER}")
print(f"📦 Modelo            : {MODEL_PATH}")
print(f"🗂️  Label mapping      : {LABEL_MAPPING_PATH}")

📂 Carpeta de audios : /home/andres/Documentos/proyecto4geeks/tests/audios/no_alertables
📦 Modelo            : /home/andres/Documentos/proyecto4geeks/models/final/best_no_alertable_v4.pt
🗂️  Label mapping      : /home/andres/Documentos/proyecto4geeks/data/interim/processed_dataset/label_mapping_human_no_alertableV2.pkl


## 3 · Cargar label mapping

In [38]:
def load_label_mapping(path: Path) -> tuple[dict, dict, int]:
    """Devuelve (label2idx, idx2label, num_classes)."""
    with open(path, "rb") as f:
        payload = pickle.load(f)

    if isinstance(payload, dict) and "label2idx" in payload:
        label2idx = payload["label2idx"]
        idx2label = payload["idx2label"]
    else:
        label2idx = payload
        idx2label = {v: k for k, v in label2idx.items()}

    num_classes = len(label2idx)
    return label2idx, idx2label, num_classes


label2idx, idx2label, NUM_CLASSES = load_label_mapping(LABEL_MAPPING_PATH)

print(f"✅ {NUM_CLASSES} clases cargadas")
print("   Clases:", list(label2idx.keys()))

✅ 14 clases cargadas
   Clases: ['ambient_noise', 'another_animal', 'breathing', 'domestic_activity', 'doors', 'engine', 'human_activity', 'impact_rattle', 'instrument_music', 'movement', 'notifications', 'voice', 'water', 'weather']


## 4 · Instanciar `Preprocess` (pipeline centralizado)

En lugar de construir los transforms manualmente, se delega en `Preprocess`.
Los parámetros deben ser **idénticos** a los usados en `preprocess.py → main()`.


In [39]:
# Instanciar Preprocess con los mismos parámetros que preprocess.py → main()
# target_duration=None → sin recorte (comportamiento de inferencia)
pp_config = PreprocessConfig(
    sample_rate=SAMPLE_RATE,
    target_duration=None,       # sin padding/recorte en inferencia
    normalize_peak=True,
    peak_target=PEAK_TARGET,
    n_mels=N_MELS,
    n_mfcc=N_MFCC,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    save_audio=False,           # no guardar nada en inferencia
    save_mel=False,
    save_mfcc=False,
)

pp = Preprocess(config=pp_config)
device = pp.device

print(f"🖥️  Dispositivo: {device}")
print("✅ Preprocess instanciado — transforms listos")


🖥️  Dispositivo: cuda
✅ Preprocess instanciado — transforms listos


## 5 · Pipeline de preprocesado — delegado en `Preprocess.process_audio_file()`

`Preprocess.process_audio_file(path)` aplica exactamente el mismo pipeline
que `preprocess.py → main()`: mono → resampleo → fix_length → normalización de pico → mel + mfcc.


In [40]:
def load_and_preprocess(
    audio_path: Path,
    augment_inference: bool = AUGMENT_INFERENCE,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Wrapper que delega en Preprocess.process_audio_file().
    Devuelve (mel, mfcc, waveform) con shape:
      mel      : [1, N_MELS, T]  float32
      mfcc     : [1, N_MFCC, T]  float32
      waveform : [1, T]           float32  — necesario para modo mel_waveform
    """
    mel, mfcc = pp.process_audio_file(audio_path, augment_inference=augment_inference)
    # Obtener waveform por separado para mel_waveform
    # process_audio_file ya aplica todo el pipeline; recalculamos solo el waveform
    import soundfile as sf
    import torchaudio.transforms as T

    audio_np, sr = sf.read(str(audio_path))
    waveform = torch.tensor(audio_np, dtype=torch.float32)
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    else:
        waveform = waveform.transpose(0, 1)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.to(device)
    if sr != SAMPLE_RATE:
        resampler = T.Resample(orig_freq=sr, new_freq=SAMPLE_RATE).to(device)
        waveform = resampler(waveform)
    peak = waveform.abs().max().clamp_min(1e-8)
    waveform = (waveform / peak * PEAK_TARGET).float()

    return mel.float(), mfcc.float(), waveform


print("✅ load_and_preprocess() listo (delega en Preprocess)")

✅ load_and_preprocess() listo (delega en Preprocess)


## 6 · Cargar modelo

In [41]:
def detect_mode_from_state_dict(state_dict: dict) -> str:
    keys = set(state_dict.keys())
    has_mel      = any(k.startswith("cnn_mel.")      for k in keys)
    has_mfcc     = any(k.startswith("cnn_mfcc.")     for k in keys)
    has_waveform = any(k.startswith("cnn_wave.") for k in keys)

    if has_mel and has_waveform:
        return "mel_waveform"
    elif has_mel and has_mfcc:
        return "mel_mfcc"
    elif has_mel:
        return "mel_only"
    elif has_mfcc:
        return "mfcc_only"
    else:
        raise ValueError("No se encontraron keys cnn_mel.*, cnn_mfcc.* ni waveform_cnn.* en el state_dict.")


def load_model(model_path: Path, num_classes: int, dropout: float) -> tuple[ImprovedMFCCCNN, str]:
    checkpoint = torch.load(model_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state" in checkpoint:
        state_dict = checkpoint["model_state"]
        epoch_info = checkpoint.get("epoch", "?")
        acc_info   = checkpoint.get("best_acc", "?")
        print(f"   📌 Checkpoint — epoch: {epoch_info}  |  best_acc: {acc_info}")
    else:
        state_dict = checkpoint

    detected_mode = detect_mode_from_state_dict(state_dict)
    print(f"   🔍 Modo detectado automáticamente: {detected_mode}")

    model = ImprovedMFCCCNN(num_classes=num_classes, dropout=dropout, mode=detected_mode).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, detected_mode


model, MODE = load_model(MODEL_PATH, NUM_CLASSES, DROPOUT)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Modelo cargado  |  {total_params:,} parámetros  |  modo: {MODE}")

   🔍 Modo detectado automáticamente: mel_waveform
✅ Modelo cargado  |  1,844,124 parámetros  |  modo: mel_waveform


## 7 · Función de predicción

In [42]:
@torch.inference_mode()
def predict(audio_path: Path, top_k: int = TOP_K) -> dict:
    """
    Retorna un dict con:
      - filename    : nombre del archivo
      - prediction  : clase predicha (bool — True=alertable, False=no alertable)
      - confidence  : probabilidad de la clase predicha (float)
      - top_k       : lista de (clase, prob) para las top_k predicciones
      - logits_raw  : tensor de logits (para debugging)
    """
    mel, mfcc, waveform = load_and_preprocess(audio_path)

    # Añadir dimensión de batch → [1, 1, F, T] / [1, 1, T]
    mel_b      = mel.unsqueeze(0)
    mfcc_b     = mfcc.unsqueeze(0)
    waveform_b = waveform.unsqueeze(0)

    if MODE == "mel_only":
        logits = model(mel=mel_b)
    elif MODE == "mel_mfcc":
        logits = model(mel=mel_b, mfcc=mfcc_b)
    elif MODE == "mel_waveform":
        logits = model(mel=mel_b, waveform=waveform_b)
    else:
        raise ValueError(f"Modo desconocido: {MODE}")

    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu()

    top_probs, top_idxs = probs.topk(min(top_k, len(idx2label)))

    pred_idx   = int(top_idxs[0])
    pred_label = idx2label[pred_idx]
    pred_conf  = float(top_probs[0])

    top_list = [(idx2label[int(i)], float(p)) for i, p in zip(top_idxs, top_probs)]

    return {
        "filename"   : audio_path.name,
        "prediction" : pred_label,
        "confidence" : pred_conf,
        "top_k"      : top_list,
        "logits_raw" : logits.squeeze(0).cpu(),
    }


print("✅ Función predict() lista")

✅ Función predict() lista


## 8 · Probar un audio individual (opcional)

In [43]:
# ── Cambia esto al archivo que quieras probar ───────────────────────────
# SINGLE_AUDIO = Path("/ruta/al/audio.wav")
# ────────────────────────────────────────────────────────────────────────

# Demo: coge el primer audio de la carpeta si existe
audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if audios:
    SINGLE_AUDIO = audios[0]
    result = predict(SINGLE_AUDIO)

    print(f"\n🔊 Archivo    : {result['filename']}")
    print(f"🏆 Predicción : {result['prediction']}")
    print(f"📊 Confianza  : {result['confidence']:.2%}")
    print(f"\n📋 Top-{TOP_K}:")
    for rank, (label, prob) in enumerate(result["top_k"], 1):
        bar = "█" * int(prob * 30)
        print(f"  {rank}. {label:<25} {prob:.2%}  {bar}")
else:
    print(f"⚠️  No hay audios .wav/.mp3 en {AUDIO_FOLDER}")


🔊 Archivo    : Alesis-Fusion-Clean-Guitar-C3.wav
🏆 Predicción : instrument_music
📊 Confianza  : 62.45%

📋 Top-5:
  1. instrument_music          62.45%  ██████████████████
  2. notifications             10.05%  ███
  3. domestic_activity         9.24%  ██
  4. engine                    6.54%  █
  5. impact_rattle             5.63%  █


## 9 · Inferencia por lotes sobre toda la carpeta

In [44]:
from tqdm import tqdm  # en vez de from tqdm.notebook import tqdm

audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if not audios:
    raise FileNotFoundError(f"No se encontraron audios en {AUDIO_FOLDER}")

print(f"📂 {len(audios)} audios encontrados en {AUDIO_FOLDER}\n")

rows = []
errors = []

for audio_path in tqdm(audios, desc="Procesando audios"):
    try:
        result = predict(audio_path)
        row = {
            "filename"   : result["filename"],
            "prediction" : result["prediction"],
            "confidence" : result["confidence"],
        }
        # Añadir columna por cada clase del top-k
        for label, prob in result["top_k"]:
            row[f"prob_{label}"] = round(prob, 4)
        rows.append(row)
    except Exception as e:
        errors.append({"filename": audio_path.name, "error": str(e)})
        print(f"❌ Error en {audio_path.name}: {e}")

results_df = pd.DataFrame(rows)

print(f"\n✅ Procesados: {len(rows)}  |  Errores: {len(errors)}")
results_df.head(20)

📂 17 audios encontrados en /home/andres/Documentos/proyecto4geeks/tests/audios/no_alertables



Procesando audios: 100%|██████████| 17/17 [00:01<00:00, 11.68it/s]


✅ Procesados: 17  |  Errores: 0


,filename,prediction,confidence,prob_instrument_music,prob_notifications,prob_another_animal,prob_domestic_activity,prob_engine,prob_impact_rattle,prob_human_activity,prob_breathing,prob_weather,prob_voice,prob_doors,prob_ambient_noise,prob_movement,prob_water
0,Alesis-Fusion-Clean-Guitar-C3.wav,instrument_music,0.844206,0.8442,0.1023,0.0117,0.0089,0.0082,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Alesis-Fusion-Steel-String-Guitar-C4.wav,instrument_music,0.930282,0.9303,0.0583,NaN,0.0011,NaN,0.0082,0.0006,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Bowed-Bass-C2.wav,engine,0.716293,NaN,0.0059,0.0172,0.1987,0.7163,0.0510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,cat1.wav,another_animal,0.988598,NaN,NaN,0.9886,0.0022,NaN,NaN,0.0014,0.0029,0.0015,NaN,NaN,NaN,NaN,NaN
4,cat10.wav,instrument_music,0.937876,0.9379,0.0016,0.0121,NaN,NaN,NaN,0.0458,NaN,NaN,0.0019,NaN,NaN,NaN,NaN
5,distant-woman-talks-of-love-in-russian-echoing...,voice,0.752038,0.0994,NaN,0.0342,0.0042,NaN,NaN,0.0941,NaN,NaN,0.7520,NaN,NaN,NaN,NaN
6,dragon-studio-door-opening-454242.mp3,doors,0.241796,NaN,NaN,NaN,0.1792,NaN,0.1547,NaN,0.0862,NaN,NaN,0.2418,0.0878,NaN,NaN
7,dragon-studio-footsteps-on-the-nature-trail-41...,movement,0.351029,NaN,NaN,NaN,0.1328,NaN,0.1848,NaN,NaN,NaN,0.0499,NaN,NaN,0.3510,0.1352
8,dragon-studio-heavy-door-unlocking-515258 (1).mp3,doors,0.506087,NaN,NaN,NaN,0.1034,NaN,0.0925,NaN,NaN,NaN,NaN,0.5061,0.0690,NaN,0.0518
9,dragon-studio-heavy-door-unlocking-515258.mp3,domestic_activity,0.339618,NaN,NaN,0.0680,0.3396,NaN,0.2023,NaN,NaN,NaN,NaN,0.1759,NaN,NaN,0.0822


## 10 · Resumen de predicciones

In [45]:
if not results_df.empty:
    summary = (
        results_df
        .groupby("prediction")
        .agg(
            count=("filename", "count"),
            avg_confidence=("confidence", "mean"),
        )
        .sort_values("count", ascending=False)
        .reset_index()
    )
    summary["avg_confidence"] = summary["avg_confidence"].map("{:.2%}".format)
    print("📊 Distribución de predicciones:")
    display(summary)

    # Archivos con confianza baja (puede necesitar revisión)
    CONFIDENCE_THRESHOLD = 0.50
    low_conf = results_df[results_df["confidence"] < CONFIDENCE_THRESHOLD]
    if not low_conf.empty:
        print(f"\n⚠️  {len(low_conf)} audios con confianza < {CONFIDENCE_THRESHOLD:.0%}:")
        display(low_conf[["filename", "prediction", "confidence"]])

📊 Distribución de predicciones:


,prediction,count,avg_confidence
0,instrument_music,4,85.68%
1,human_activity,2,43.11%
2,domestic_activity,2,40.63%
3,engine,2,46.51%
4,doors,2,37.39%
5,notifications,2,65.90%
6,another_animal,1,98.86%
7,movement,1,35.10%
8,voice,1,75.20%



⚠️  7 audios con confianza < 50%:


,filename,prediction,confidence
6,dragon-studio-door-opening-454242.mp3,doors,0.241796
7,dragon-studio-footsteps-on-the-nature-trail-41...,movement,0.351029
9,dragon-studio-heavy-door-unlocking-515258.mp3,domestic_activity,0.339618
10,dragon-studio-open-door-stock-sfx-454246 (1).mp3,human_activity,0.175241
11,dragon-studio-open-door-stock-sfx-454246.mp3,domestic_activity,0.473055
12,freesound_community-print-shop-printer-3-23680...,engine,0.213919
13,messenger-notification.wav,notifications,0.407681


## 11 · Exportar resultados a CSV (opcional)

In [46]:
OUTPUT_CSV = AUDIO_FOLDER / "predictions.csv"

if not results_df.empty:
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Resultados guardados en: {OUTPUT_CSV}")
else:
    print("⚠️  No hay resultados para exportar.")

✅ Resultados guardados en: /home/andres/Documentos/proyecto4geeks/tests/audios/no_alertables/predictions.csv
